<a href="https://colab.research.google.com/github/ranjetmahato416/-Lung-CT-Image-Classification-Using-Public-Medical-Imaging-Data-/blob/main/Desktop/Colab_Notebook/Notebook/Notebook_13_Comparative_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 13 — Comparative Evaluation of CNN Architectures

This notebook compares the selected EfficientNetB0, DenseNet121 and ResNet50 models developed for binary lung nodule malignancy classification.

Each architecture was trained using the same patient-level training, validation and test splits, image size, class weighting, augmentation strategy and evaluation protocol.

For each architecture, the frozen-backbone baseline and fine-tuned model were compared using validation PR-AUC. Only the selected configuration was evaluated on the untouched test set.

The objectives of this notebook are to:

1. consolidate the results from all three architectures;
2. compare discrimination using test ROC-AUC and PR-AUC;
3. compare classification performance at predefined operating thresholds;
4. select the final model for explainability analysis using Grad-CAM.

In [1]:
from pathlib import Path

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

##3. Mount Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


##4. Define model directories

In [3]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Dissertation"
)

MODEL_ROOT = (
    PROJECT_ROOT / "Models"
)

EFFICIENTNET_ROOT = (
    MODEL_ROOT
    / "EfficientNetB0_FineTune_Last30"
)

DENSENET_ROOT = (
    MODEL_ROOT
    / "DenseNet121"
    / "Corrected_Run_01"
)

RESNET_ROOT = (
    MODEL_ROOT
    / "ResNet50"
    / "Corrected_Run_01"
)

COMPARISON_ROOT = (
    MODEL_ROOT
    / "Model_Comparison"
)

COMPARISON_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Comparison output directory:")
print(COMPARISON_ROOT)

Comparison output directory:
/content/drive/MyDrive/Dissertation/Models/Model_Comparison


##5. Define summary-file paths

In [4]:
EFFICIENTNET_SUMMARY_PATH = (
    EFFICIENTNET_ROOT
    / "model_summary.csv"
)

DENSENET_SUMMARY_PATH = (
    DENSENET_ROOT
    / "Results"
    / "densenet121_experiment_summary.csv"
)

RESNET_SUMMARY_PATH = (
    RESNET_ROOT
    / "Results"
    / "resnet50_experiment_summary.csv"
)

for model_name, path in {
    "EfficientNetB0": EFFICIENTNET_SUMMARY_PATH,
    "DenseNet121": DENSENET_SUMMARY_PATH,
    "ResNet50": RESNET_SUMMARY_PATH
}.items():

    print(
        model_name,
        "summary exists:",
        path.exists(),
        "—",
        path
    )

EfficientNetB0 summary exists: True — /content/drive/MyDrive/Dissertation/Models/EfficientNetB0_FineTune_Last30/model_summary.csv
DenseNet121 summary exists: True — /content/drive/MyDrive/Dissertation/Models/DenseNet121/Corrected_Run_01/Results/densenet121_experiment_summary.csv
ResNet50 summary exists: True — /content/drive/MyDrive/Dissertation/Models/ResNet50/Corrected_Run_01/Results/resnet50_experiment_summary.csv


##6. Inspect summary schemas

In [5]:
efficientnet_summary = pd.read_csv(
    EFFICIENTNET_SUMMARY_PATH
)

densenet_summary = pd.read_csv(
    DENSENET_SUMMARY_PATH
)

resnet_summary = pd.read_csv(
    RESNET_SUMMARY_PATH
)

print("EfficientNet columns:")
print(efficientnet_summary.columns.tolist())

print("\nDenseNet columns:")
print(densenet_summary.columns.tolist())

print("\nResNet columns:")
print(resnet_summary.columns.tolist())

display(efficientnet_summary)
display(densenet_summary)
display(resnet_summary)

EfficientNet columns:
['SelectedModel', 'ValidationROC_AUC', 'ValidationPR_AUC', 'TestROC_AUC', 'TestPR_AUC', 'DefaultThreshold', 'ValidationF1Threshold', 'HighRecallThreshold']

DenseNet columns:
['Selected Model', 'Baseline Validation ROC-AUC', 'Baseline Validation PR-AUC', 'Fine-Tuned Validation ROC-AUC', 'Fine-Tuned Validation PR-AUC', 'Validation F1 Threshold', 'Validation High-Recall Threshold', 'Test ROC-AUC', 'Test PR-AUC']

ResNet columns:
['Selected Model', 'Baseline Validation ROC-AUC', 'Baseline Validation PR-AUC', 'Baseline Validation F1', 'Fine-Tuned Validation ROC-AUC', 'Fine-Tuned Validation PR-AUC', 'Fine-Tuned Validation F1', 'Selected Validation F1 Threshold', 'Selected High-Recall Threshold', 'Test ROC-AUC', 'Test PR-AUC']


,SelectedModel,ValidationROC_AUC,ValidationPR_AUC,TestROC_AUC,TestPR_AUC,DefaultThreshold,ValidationF1Threshold,HighRecallThreshold
0,Fine-Tuned EfficientNetB0,0.850649,0.570277,0.830252,0.633736,0.5,0.464395,0.327528


,Selected Model,Baseline Validation ROC-AUC,Baseline Validation PR-AUC,Fine-Tuned Validation ROC-AUC,Fine-Tuned Validation PR-AUC,Validation F1 Threshold,Validation High-Recall Threshold,Test ROC-AUC,Test PR-AUC
0,Fine-Tuned DenseNet121,0.814935,0.489087,0.861586,0.587677,0.669041,0.445648,0.855333,0.588136


,Selected Model,Baseline Validation ROC-AUC,Baseline Validation PR-AUC,Baseline Validation F1,Fine-Tuned Validation ROC-AUC,Fine-Tuned Validation PR-AUC,Fine-Tuned Validation F1,Selected Validation F1 Threshold,Selected High-Recall Threshold,Test ROC-AUC,Test PR-AUC
0,Fine-Tuned ResNet50,0.797847,0.467531,0.478261,0.797163,0.497598,0.489796,0.600719,0.113742,0.822883,0.562283


##7. Build a standardized comparison table

In [6]:
def get_first_available(
    dataframe,
    possible_columns
):

    for column in possible_columns:

        if column in dataframe.columns:

            return dataframe.loc[
                dataframe.index[0],
                column
            ]

    return np.nan

In [7]:
efficientnet_row = {
    "Architecture": "EfficientNetB0",

    "Selected Configuration":
        efficientnet_summary.loc[0, "SelectedModel"],

    "Validation ROC-AUC":
        efficientnet_summary.loc[0, "ValidationROC_AUC"],

    "Validation PR-AUC":
        efficientnet_summary.loc[0, "ValidationPR_AUC"],

    "Test ROC-AUC":
        efficientnet_summary.loc[0, "TestROC_AUC"],

    "Test PR-AUC":
        efficientnet_summary.loc[0, "TestPR_AUC"]
}

In [8]:
architecture_comparison = pd.DataFrame(
    [
        efficientnet_row,

        {
            "Architecture": "DenseNet121",

            "Selected Configuration":
                densenet_summary.loc[
                    0,
                    "Selected Model"
                ],

            "Validation ROC-AUC":
                densenet_summary.loc[
                    0,
                    "Fine-Tuned Validation ROC-AUC"
                ],

            "Validation PR-AUC":
                densenet_summary.loc[
                    0,
                    "Fine-Tuned Validation PR-AUC"
                ],

            "Test ROC-AUC":
                densenet_summary.loc[
                    0,
                    "Test ROC-AUC"
                ],

            "Test PR-AUC":
                densenet_summary.loc[
                    0,
                    "Test PR-AUC"
                ]
        },

        {
            "Architecture": "ResNet50",

            "Selected Configuration":
                resnet_summary.loc[
                    0,
                    "Selected Model"
                ],

            "Validation ROC-AUC":
                resnet_summary.loc[
                    0,
                    "Fine-Tuned Validation ROC-AUC"
                ],

            "Validation PR-AUC":
                resnet_summary.loc[
                    0,
                    "Fine-Tuned Validation PR-AUC"
                ],

            "Test ROC-AUC":
                resnet_summary.loc[
                    0,
                    "Test ROC-AUC"
                ],

            "Test PR-AUC":
                resnet_summary.loc[
                    0,
                    "Test PR-AUC"
                ]
        }
    ]
)

display(architecture_comparison)

,Architecture,Selected Configuration,Validation ROC-AUC,Validation PR-AUC,Test ROC-AUC,Test PR-AUC
0,EfficientNetB0,Fine-Tuned EfficientNetB0,0.850649,0.570277,0.830252,0.633736
1,DenseNet121,Fine-Tuned DenseNet121,0.861586,0.587677,0.855333,0.588136
2,ResNet50,Fine-Tuned ResNet50,0.797163,0.497598,0.822883,0.562283


##8. Validate the comparison table

In [9]:
required_metric_columns = [
    "Validation ROC-AUC",
    "Validation PR-AUC",
    "Test ROC-AUC",
    "Test PR-AUC"
]

missing_metrics = (
    architecture_comparison[
        required_metric_columns
    ]
    .isna()
)

if missing_metrics.any().any():

    print(
        "Some values could not be found "
        "automatically:"
    )

    display(
        architecture_comparison
    )

    print(
        "\nReview the CSV column names printed "
        "in Section 6."
    )

else:

    print(
        "All model-level metrics were loaded "
        "successfully."
    )

All model-level metrics were loaded successfully.


##9. Rank models

In [10]:
# ============================================================
# Rank Architectures Using VALIDATION PR-AUC
# ============================================================

architecture_comparison[
    "Validation PR-AUC Rank"
] = (
    architecture_comparison[
        "Validation PR-AUC"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype("Int64")
)

architecture_comparison[
    "Validation ROC-AUC Rank"
] = (
    architecture_comparison[
        "Validation ROC-AUC"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype("Int64")
)

architecture_comparison = (
    architecture_comparison
    .sort_values(
        by="Validation PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    architecture_comparison
)

,Architecture,Selected Configuration,Validation ROC-AUC,Validation PR-AUC,Test ROC-AUC,Test PR-AUC,Validation PR-AUC Rank,Validation ROC-AUC Rank
0,DenseNet121,Fine-Tuned DenseNet121,0.861586,0.587677,0.855333,0.588136,1,1
1,EfficientNetB0,Fine-Tuned EfficientNetB0,0.850649,0.570277,0.830252,0.633736,2,2
2,ResNet50,Fine-Tuned ResNet50,0.797163,0.497598,0.822883,0.562283,3,3


##10. Select the final architecture

In [11]:
# ============================================================
# Select Final Architecture Using Validation PR-AUC
# ============================================================

best_model_row = (
    architecture_comparison
    .iloc[0]
)

FINAL_ARCHITECTURE = (
    best_model_row[
        "Architecture"
    ]
)

FINAL_CONFIGURATION = (
    best_model_row[
        "Selected Configuration"
    ]
)

FINAL_VALIDATION_ROC_AUC = float(
    best_model_row[
        "Validation ROC-AUC"
    ]
)

FINAL_VALIDATION_PR_AUC = float(
    best_model_row[
        "Validation PR-AUC"
    ]
)

FINAL_TEST_ROC_AUC = float(
    best_model_row[
        "Test ROC-AUC"
    ]
)

FINAL_TEST_PR_AUC = float(
    best_model_row[
        "Test PR-AUC"
    ]
)

print(
    "Final selected architecture:",
    FINAL_ARCHITECTURE
)

print(
    "Selected configuration:",
    FINAL_CONFIGURATION
)

print(
    "Validation ROC-AUC:",
    FINAL_VALIDATION_ROC_AUC
)

print(
    "Validation PR-AUC:",
    FINAL_VALIDATION_PR_AUC
)

print(
    "Final test ROC-AUC:",
    FINAL_TEST_ROC_AUC
)

print(
    "Final test PR-AUC:",
    FINAL_TEST_PR_AUC
)

Final selected architecture: DenseNet121
Selected configuration: Fine-Tuned DenseNet121
Validation ROC-AUC: 0.8615857826384143
Validation PR-AUC: 0.5876770576684937
Final test ROC-AUC: 0.855332902391726
Final test PR-AUC: 0.5881362975400494


##11. Save the architecture comparison

In [12]:
ARCHITECTURE_COMPARISON_PATH = (
    COMPARISON_ROOT
    / "architecture_comparison.csv"
)

architecture_comparison.to_csv(
    ARCHITECTURE_COMPARISON_PATH,
    index=False
)

print(
    "Saved architecture comparison:",
    ARCHITECTURE_COMPARISON_PATH
)

Saved architecture comparison: /content/drive/MyDrive/Dissertation/Models/Model_Comparison/architecture_comparison.csv


##12. Save final model selection

In [13]:
final_model_selection = {
    "FinalArchitecture":
        FINAL_ARCHITECTURE,

    "SelectedConfiguration":
        str(FINAL_CONFIGURATION),

    "PrimarySelectionMetric":
        "Validation PR-AUC",

    "ValidationROCAUC":
        FINAL_VALIDATION_ROC_AUC,

    "ValidationPRAUC":
        FINAL_VALIDATION_PR_AUC,

    "TestROCAUC":
        FINAL_TEST_ROC_AUC,

    "TestPRAUC":
        FINAL_TEST_PR_AUC
}

FINAL_SELECTION_PATH = (
    COMPARISON_ROOT
    / "final_model_selection.json"
)

with open(
    FINAL_SELECTION_PATH,
    "w"
) as file:

    json.dump(
        final_model_selection,
        file,
        indent=4
    )

print(
    "Updated final model selection:",
    FINAL_SELECTION_PATH
)

Updated final model selection: /content/drive/MyDrive/Dissertation/Models/Model_Comparison/final_model_selection.json
